In [9]:
from dotenv import load_dotenv
from langfuse import get_client, Evaluation
from datetime import datetime
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from getpass import getpass
import os
import re

In [10]:
load_dotenv() # TODO : pour recuperer les infos utiles a Langfuse

True

In [11]:
dataset_name = "small_dataset_to_test"
langfuse = get_client()

langfuse.create_dataset(
    name=dataset_name,
    description="Dataset pour prendre en main cette fonctionnalité",
    metadata={
        "author": "Saskya",
        "date": "2026-03-27",
        "type": "test"
    }
)

# TODO : parser le googlesheet pour creer automatiquement dataset
items = [
    {
        "input": {
            "query": "C'est quoi OHS ?"
        },
        "metadata": {
            "expected_doc_id": "108a7403f048034b375564dcef3a9b1b" # TODO : ajouter dans metadata des chunks lors de l'indexation avec Qdrant ?
        }
    },
    {
        "input": {
            "query": "Pourquoi la commission des prix de la recherche de l'EPFL est composée obligatoirement de 16 membres ?"
        },
        "metadata": {
            "expected_doc_id": "33f4883d8d8c56a98d3ae27c80f7e1be"
        }
    }
]

for item in items:
    langfuse.create_dataset_item(
        dataset_name=dataset_name,
        input=item.get("input"),
        metadata=item.get("metadata")
    )

# TODO : si documents de nouveau telecharges alors id vont changer, comment fixer ca et mettre une reference stable dans le dataset de dev/test ???

In [17]:
# TODO : refactor dans un fichier utils car copie de prompter

COLLECTION_NAME = "basic_split_pages_all_formats"
K = 30

if not os.getenv("OPENAI_API_KEY_EMBEDDINGS"):
    os.environ["OPENAI_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_EMBEDDINGS")
)

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME, # TODO
    url="http://localhost:6333",
)

In [18]:
def retrieval_task(*, item, **kwargs):
    input = item.input
    query = input.get("query")
    hits = qdrant.similarity_search_with_score(query, k=K)

    retrieved_doc_ids = []
    retrieved_scores = []

    for hit in hits:
        document, score = hit
        source = document.metadata.get("source")
        doc_id = re.search(r"^data/(.+)\.", source).group(1) # TODO : update si collection Qdrant modifiee
        retrieved_doc_ids.append(doc_id)
        retrieved_scores.append(score)

    return {
        "retrieved_doc_ids": retrieved_doc_ids,
        "retrieved_scores": retrieved_scores
    }

In [14]:
def hit_at_1_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    retrieved_scores = output.get("retrieved_scores")
    value = 1.0 if expected_doc_id == retrieved_doc_ids[0] else 0.0
    return Evaluation(
        name="hit_at_1",
        value=value,
        comment=f"expected_doc_id={expected_doc_id}, top1={retrieved_doc_ids[0]} with score {retrieved_scores[0]}", # TODO : mettre les scores autrement pour pouvoir average dessus ?
    )

def hit_at_5_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    retrieved_scores = output.get("retrieved_scores")
    value = 1.0 if expected_doc_id in retrieved_doc_ids[:5] else 0.0
    return Evaluation(
        name="hit_at_5",
        value=value,
        comment=f"expected_doc_id={expected_doc_id}, top5={retrieved_doc_ids[:5]} with scores {retrieved_scores[:5]}",
    )

def mrr_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == expected_doc_id:
            reciprocal_rank = 1.0 / rank
            break
    return Evaluation(
        name="mrr_doc",
        value=reciprocal_rank,
        comment=f"expected_doc_id={expected_doc_id}, retrieved={retrieved_doc_ids}",
    )

In [15]:
def avg_hit_at_1(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "hit_at_1"
    ]
    if not vals:
        return Evaluation(name="avg_hit_at_1", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_hit_at_1", value=avg)

def avg_hit_at_5(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "hit_at_5"
    ]
    if not vals:
        return Evaluation(name="avg_hit_at_5", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_hit_at_5", value=avg)

def avg_mrr_doc(*, item_results, **kwargs):
    vals = [
        ev.value for result in item_results
        for ev in result.evaluations
        if ev.name == "mrr_doc"
    ]
    if not vals:
        return Evaluation(name="avg_mrr_doc", value=None)
    avg = sum(vals) / len(vals)
    return Evaluation(name="avg_mrr_doc", value=avg)

In [19]:
dataset = langfuse.get_dataset(dataset_name)
run_name = f"test_experiment_functionality_{datetime.now().isoformat()}"

result = dataset.run_experiment(
    name="test_experiments",
    run_name=run_name,
    description="Test experiment to score retrieval task (with dummy dataset)",
    task=retrieval_task,
    evaluators=[hit_at_1_evaluator, hit_at_5_evaluator, mrr_doc_evaluator],
    run_evaluators=[avg_hit_at_1, avg_hit_at_5, avg_mrr_doc],
    metadata={
        "collection_name": COLLECTION_NAME, # TODO
        "top_k": K, # TODO
        "chunking_strategy": "per_page" # TODO
    }
)

print(result.format())

Individual Results: Hidden (2 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: test_experiments
📋 Run name: test_experiment_functionality_2026-03-27T14:36:39.572923 - Test experiment to score retrieval task (with dummy dataset)\n2 items\nEvaluations:\n  • hit_at_5\n  • hit_at_1\n  • mrr_doc\n\nAverage Scores:\n  • hit_at_5: 0.000\n  • hit_at_1: 0.000\n  • mrr_doc: 0.000\n\nRun Evaluations:\n  • avg_hit_at_1: 0.000\n  • avg_hit_at_5: 0.000\n  • avg_mrr_doc: 0.000\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmn8y0dyi001gnw07dc1dlrqt/runs/ff146efe-dfed-424a-b98c-10d30dde36b2
